# Dialogue Summarization Project - Phase 3

## Transformer-Based Summarization with BART

In Phase 2, we created a classical TF-IDF extractive baseline.

In Phase 3, we use a transformer model for abstractive dialogue summarization.

Model used:

- `facebook/bart-base`

This notebook is beginner-friendly and designed to run in Google Colab.

## 1. Install Required Libraries

If you are using Google Colab, run this cell once.

The `evaluate` library is used for ROUGE metrics, and `accelerate` helps the Hugging Face Trainer work smoothly with GPU.

In [ ]:
# For Google Colab, uncomment and run this line if needed.
# !pip install datasets transformers evaluate accelerate torch sentencepiece pandas matplotlib

## 2. Import Libraries

We import dataset tools, transformer tools, evaluation tools, and our own helper functions.

In [ ]:
import os
import sys

import pandas as pd
import torch
from datasets import load_dataset

# Add src folder first so our local evaluate.py is used.
sys.path.insert(0, "src")

from train_transformer import (
    MODEL_NAME,
    MAX_INPUT_LENGTH,
    MAX_TARGET_LENGTH,
    NUM_TRAIN_EPOCHS,
    get_device_name,
    load_model_and_tokenizer,
    prepare_tokenized_dataset,
    create_trainer,
    plot_training_curve,
    evaluate_on_test_set,
    create_comparison_table,
)

# Create output folders.
os.makedirs("models/transformer", exist_ok=True)
os.makedirs("models/transformer-checkpoints", exist_ok=True)
os.makedirs("results", exist_ok=True)

print("Libraries imported successfully!")
print("Device:", get_device_name())
print("Training epochs:", NUM_TRAIN_EPOCHS)

## 3. Load the SAMSum Dataset

We use the same SAMSum dataset from earlier phases.

The dataset has train, validation, and test splits.

In [ ]:
# Load dataset from Hugging Face.
dataset = load_dataset("knkarthick/samsum")

# Display dataset structure.
dataset

In [ ]:
# Show one example dialogue and summary.
sample = dataset["train"][0]

print("Dialogue:\n")
print(sample["dialogue"])

print("\nHuman Summary:\n")
print(sample["summary"])

## 4. Load BART Model and Tokenizer

A tokenizer converts text into numbers that the model can understand.

The model then learns to generate summary token IDs from dialogue token IDs.

In [ ]:
# Load facebook/bart-base tokenizer and model.
tokenizer, model = load_model_and_tokenizer(MODEL_NAME)

print("Loaded model:", MODEL_NAME)
print("Maximum input length:", MAX_INPUT_LENGTH)
print("Maximum summary length:", MAX_TARGET_LENGTH)

## 5. Tokenization Example

Here we tokenize one dialogue to see what the tokenizer produces.

The model does not read raw text directly. It reads token IDs.

In [ ]:
# Add a task prefix to make the instruction clear.
example_input = "summarize: " + sample["dialogue"]

# Tokenize the example.
tokenized_example = tokenizer(
    example_input,
    max_length=MAX_INPUT_LENGTH,
    truncation=True,
)

print("First 20 token IDs:")
print(tokenized_example["input_ids"][:20])

print("\nDecoded text from token IDs:")
print(tokenizer.decode(tokenized_example["input_ids"][:80]))

## 6. Preprocess Dataset

The preprocessing step tokenizes dialogues and summaries.

To keep this notebook easy to run, the helper function uses a small subset by default:

- 500 training samples
- 100 validation samples
- 20 test samples

You can increase these values later inside `src/train_transformer.py`.

In [ ]:
# Prepare small raw dataset and tokenized dataset.
small_dataset, tokenized_dataset = prepare_tokenized_dataset(dataset, tokenizer)

print(tokenized_dataset)
print("Train samples:", len(tokenized_dataset["train"]))
print("Validation samples:", len(tokenized_dataset["validation"]))
print("Test samples:", len(tokenized_dataset["test"]))

## 7. Create Trainer

Hugging Face `Seq2SeqTrainer` handles the training loop for us.

It supports:

- GPU training if available
- Evaluation after each epoch
- Saving checkpoints
- Generating summaries during evaluation
- ROUGE metric calculation

In [ ]:
# Create Trainer object.
trainer = create_trainer(
    tokenizer=tokenizer,
    model=model,
    tokenized_dataset=tokenized_dataset,
    output_dir="models/transformer-checkpoints",
)

trainer

## 8. Train the Model

This step fine-tunes BART on SAMSum.

Training can take time. It is much faster with a GPU in Google Colab.

Week 3 requires at least 3 epochs, so this notebook uses the `NUM_TRAIN_EPOCHS` value from `src/train_transformer.py`.

In [ ]:
# Start training.
# Checkpoints are saved inside models/transformer-checkpoints/.
trainer.train()

## 9. Evaluate on Validation Data

After training, we evaluate the model on validation data and calculate ROUGE scores.

In [ ]:
# Evaluate the model.
validation_metrics = trainer.evaluate()

# Convert metrics dictionary to a table.
metrics_df = pd.DataFrame([validation_metrics])
metrics_df

## 10. Save the Model

We save the final trained model and tokenizer in `models/transformer/`.

Later, inference code can load this folder directly.

In [ ]:
# Save trained model and tokenizer.
trainer.save_model("models/transformer")
tokenizer.save_pretrained("models/transformer")

print("Model saved in models/transformer/")

In [ ]:
# CRITICAL: Verify that the saved model is correctly loaded.
# This ensures evaluation uses the FINE-TUNED model, not the base facebook/bart-base.

print("\n" + "="*60)
print("MODEL VERIFICATION CHECK")
print("="*60)

# Reload the model fresh from disk to simulate a new session
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

saved_tokenizer = AutoTokenizer.from_pretrained("models/transformer")
saved_model = AutoModelForSeq2SeqLM.from_pretrained("models/transformer")

print(f"✓ Model config origin: {saved_model.config._name_or_path}")
print(f"✓ Tokenizer vocab size: {len(saved_tokenizer)}")
print(f"✓ Model has {sum(p.numel() for p in saved_model.parameters())} total parameters")
print(f"✓ Fine-tuned model successfully loaded from: models/transformer")
print("="*60 + "\n")

# Use the reloaded model for the following evaluation
model = saved_model
tokenizer = saved_tokenizer

## 11. Save Validation Metrics

The metrics are saved as a CSV file so they can be used in reports.

In [ ]:
# Save validation metrics separately.
# Validation metrics show training progress; final reporting uses the test set.
metrics_df.to_csv("results/transformer_validation_metrics.csv", index=False)

print("Saved results/transformer_validation_metrics.csv")

## 12. Plot Training and Validation Loss Curve

A loss curve helps us see whether the model improved during training and whether validation loss behaved reasonably.

In [ ]:
# Save training loss graph.
plot_training_curve(
    trainer=trainer,
    output_path="results/training_curve.png",
)

print("Saved results/training_curve.png")

## 13. Evaluate on Test Set and Generate Sample Summaries

Now we generate summaries for the test set, compute ROUGE, and save 5-10 sample outputs for the report.

The files saved are:

- `results/transformer_metrics.csv`
- `results/transformer_test_predictions.csv`
- `results/transformer_sample_predictions.csv`

In [ ]:
# Evaluate the final model on the test set.
test_metrics_df, test_predictions_df, sample_predictions_df = evaluate_on_test_set(
    trainer=trainer,
    tokenizer=tokenizer,
    original_test_dataset=small_dataset["test"],
    tokenized_test_dataset=tokenized_dataset["test"],
    metrics_output_path="results/transformer_metrics.csv",
    predictions_output_path="results/transformer_test_predictions.csv",
    sample_output_path="results/transformer_sample_predictions.csv",
)

sample_predictions_df.head(10)

## 14. Create Baseline vs Transformer Comparison Table

This table compares the Week 2 TF-IDF baseline with the Week 3 fine-tuned transformer.

In [ ]:
comparison_df = create_comparison_table(
    transformer_metrics_df=test_metrics_df,
    baseline_table_path="results/comparison_table.csv",
    output_path="results/comparison_table.csv",
)

comparison_df

## 15. Use the Inference Pipeline

The inference helper can load the saved model and summarize a new dialogue.

In [ ]:
from inference import load_summarization_pipeline, generate_summary

# Load saved model into a summarization pipeline.
summarizer = load_summarization_pipeline("models/transformer")

# Try a new dialogue.
new_dialogue = """
A: Are you joining the project meeting today?
B: Yes, I will join at 3 PM.
A: Great. Please bring the latest report.
B: Sure, I finished it last night.
"""

summary = generate_summary(new_dialogue, summarizer)

print("Generated Summary:")
print(summary)

## 16. Phase 3 Summary

In this phase, we completed a transformer-based summarization pipeline:

- Loaded the SAMSum dataset
- Loaded `facebook/bart-base`
- Tokenized dialogues and summaries
- Used `DataCollatorForSeq2Seq`
- Trained with Hugging Face `Seq2SeqTrainer`
- Evaluated with ROUGE
- Saved checkpoints and final model
- Saved validation metrics, training curve, and sample summaries

The saved model is available in `models/transformer/`.